# Stuff
# Map-Recude

In [2]:
import os

from langchain_openai import ChatOpenAI

model = ChatOpenAI(
    model='deepseek-ai/DeepSeek-V3.2',
    base_url=os.environ.get('OPENAI_BASE_URL'),
    api_key=os.environ.get('OPENAI_API_KEY'),
    temperature=0
)

In [3]:

from langchain_community.document_loaders import WebBaseLoader

# 1. 加载数据: 一篇博客内容
loader = WebBaseLoader("https://lilianweng.github.io/posts/2023-06-23-agent/")
docs = loader.load()

USER_AGENT environment variable not set, consider setting it to identify your requests.


In [4]:
from langchain_classic.chains.summarize import load_summarize_chain

# 第一种 无法设置提示词
chain = load_summarize_chain(model, chain_type="stuff")
result = chain.invoke(docs)
result

{'input_documents': [Document(metadata={'source': 'https://lilianweng.github.io/posts/2023-06-23-agent/', 'title': "LLM Powered Autonomous Agents | Lil'Log", 'description': 'Building agents with LLM (large language model) as its core controller is a cool concept. Several proof-of-concepts demos, such as AutoGPT, GPT-Engineer and BabyAGI, serve as inspiring examples. The potentiality of LLM extends beyond generating well-written copies, stories, essays and programs; it can be framed as a powerful general problem solver.\nAgent System Overview\nIn a LLM-powered autonomous agent system, LLM functions as the agent’s brain, complemented by several key components:\n\nPlanning\n\nSubgoal and decomposition: The agent breaks down large tasks into smaller, manageable subgoals, enabling efficient handling of complex tasks.\nReflection and refinement: The agent can do self-criticism and self-reflection over past actions, learn from mistakes and refine them for future steps, thereby improving the q

In [5]:
result['output_text']

"This blog post outlines the framework for building autonomous agents powered by Large Language Models (LLMs). It identifies three core components:\n\n1.  **Planning:** The agent breaks down complex tasks (via techniques like Chain of Thought) and engages in self-reflection to learn from mistakes.\n2.  **Memory:** It utilizes short-term memory (the model's context window) and long-term memory (an external database for storing and retrieving vast information).\n3.  **Tool Use:** The agent learns to call external APIs and tools (like calculators or search engines) to extend its capabilities beyond its pre-trained knowledge.\n\nThe post reviews proof-of-concept systems like AutoGPT and case studies in scientific discovery, while also acknowledging key challenges such as finite context length, unreliable natural language interfaces, and difficulties in long-term planning."

In [9]:
# 第二种 自定义提示词是摘要为中文
from langchain_core.prompts import PromptTemplate
from langchain_classic.chains.llm import LLMChain
from langchain_classic.chains.combine_documents.stuff import StuffDocumentsChain

document_prompt = PromptTemplate(
    input_variables=["page_content"], template="{page_content}"
)
document_variable_name = "context"

prompt = PromptTemplate.from_template("""
针对下面的内容，写一个简洁的摘要:
{context}
简洁的总结摘要:""")
llm_chain = LLMChain(llm=model, prompt=prompt)
chain = StuffDocumentsChain(
    llm_chain=llm_chain,
    document_prompt=document_prompt,
    document_variable_name=document_variable_name,
)

In [10]:
result = chain.invoke(docs)
result

{'input_documents': [Document(metadata={'source': 'https://lilianweng.github.io/posts/2023-06-23-agent/', 'title': "LLM Powered Autonomous Agents | Lil'Log", 'description': 'Building agents with LLM (large language model) as its core controller is a cool concept. Several proof-of-concepts demos, such as AutoGPT, GPT-Engineer and BabyAGI, serve as inspiring examples. The potentiality of LLM extends beyond generating well-written copies, stories, essays and programs; it can be framed as a powerful general problem solver.\nAgent System Overview\nIn a LLM-powered autonomous agent system, LLM functions as the agent’s brain, complemented by several key components:\n\nPlanning\n\nSubgoal and decomposition: The agent breaks down large tasks into smaller, manageable subgoals, enabling efficient handling of complex tasks.\nReflection and refinement: The agent can do self-criticism and self-reflection over past actions, learn from mistakes and refine them for future steps, thereby improving the q

In [11]:
result['output_text']

'本文系统介绍了以大型语言模型（LLM）为核心的自主智能代理系统。文章指出，LLM 可作为代理的“大脑”，通过三大核心组件实现复杂任务处理：\n\n1. **规划**：将复杂任务分解为子目标，并通过自我反思优化决策。\n2. **记忆**：结合短期记忆（上下文学习）与长期记忆（外部向量存储），实现信息持久化与快速检索。\n3. **工具使用**：调用外部 API 获取实时信息、执行代码等，扩展模型能力。\n\n文章还分析了多个实际案例（如 AutoGPT、HuggingGPT）与实验，展示了智能代理在科学发现、模拟社会交互等场景的应用潜力，并指出当前系统在上下文长度、长期规划可靠性、自然语言接口稳定性等方面仍面临挑战。'

# Map-Reduce 处理方法
## Reduce的思路
1. 如果Map之后文档累计token数量超过了4000个，那么将递归地将文档以<=4000个token批次传递给StuffDocumentsChain
2. 一旦这些批量摘要的累计大小小于4000个token,那么全部传递

In [16]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 文本切割
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    encoding_name="cl100k_base",  # GPT-4 使用的编码器，最适合英文
    chunk_size=1000,  # 每个 chunk 的 token 数
    chunk_overlap=0  # 无重叠
)

In [17]:
docs_split = text_splitter.split_documents(docs)

In [12]:
# 第3种 map reduce
# 1. map
map_prompt = PromptTemplate.from_template("""
以下是一组文档(documents)
"{docs}"
根据这个文档列表，请给出总结摘要:
""")
map_llm_chain = LLMChain(llm=model, prompt=map_prompt)
# 2. reduce
reduce_prompt = PromptTemplate.from_template("""
以下是一组总结摘要:
"{docs}"
将这些内容提炼成一个最终，统一的总结摘要:
""")
reduce_llm_chain = LLMChain(llm=model, prompt=reduce_prompt)

In [18]:
from langchain_classic.chains.combine_documents.map_reduce import MapReduceDocumentsChain
from langchain_classic.chains.combine_documents.reduce import ReduceDocumentsChain

# 3. combine
combine_chain = StuffDocumentsChain(llm_chain=reduce_llm_chain, document_variable_name='docs')

# 4. reduce_chain
reduce_chain = ReduceDocumentsChain(
    # 最终调用的链
    combine_documents_chain=combine_chain,
    # 中间调用的链
    collapse_documents_chain=combine_chain,
    # 将文档分组的最大token
    token_max=4000
)
# 5. map_reduce_chain
map_reduce_chain = MapReduceDocumentsChain(
    llm_chain=map_llm_chain,
    reduce_documents_chain=reduce_chain,
    document_variable_name='docs',
    # 不要返回中间汇总结果
    return_intermediate_steps=False
)

C:\Users\14820\AppData\Local\Temp\ipykernel_7532\1002972140.py:8: LangChainDeprecationWarning: This class is deprecated. Please see the migration guide here for a recommended replacement: https://python.langchain.com/docs/versions/migrating_chains/map_reduce_chain/
  reduce_chain = ReduceDocumentsChain(
C:\Users\14820\AppData\Local\Temp\ipykernel_7532\1002972140.py:17: LangChainDeprecationWarning: This class is deprecated. Please see the migration guide here for a recommended replacement: https://python.langchain.com/docs/versions/migrating_chains/map_reduce_chain/
  map_reduce_chain = MapReduceDocumentsChain(


In [19]:
result = map_reduce_chain.invoke(docs_split)

D:\Work\Langchain-learn-demo\.venv\Lib\site-packages\langchain_openai\chat_models\base.py:483: UserWarning: Unexpected type for token usage: <class 'NoneType'>
  warnings.warn(f"Unexpected type for token usage: {type(new_usage)}")


In [23]:

from IPython.display import display, Markdown

display(Markdown(f"{result['output_text']}"))

**基于大型语言模型（LLM）的自主智能体系统：统一总结摘要**

本摘要系统性地阐述了以LLM为核心的自主智能体（AI Agent）的技术架构、核心能力、应用实践、当前挑战与发展方向。

**一、 核心架构与工作机制**
LLM作为智能体的“大脑”或核心控制器，通过系统性地整合**规划、记忆与工具使用**三大关键能力，使其能够处理复杂的现实世界任务。其通用工作流程遵循“**规划-调用-整合**”范式：
1.  **规划**：将复杂任务分解为有序子目标（通过思维链、思维树等方法），并通过自我反思（如ReAct框架）从经验中学习与修正。
2.  **工具使用**：调用外部API、计算工具或专家模型（通过函数调用、微调等方式），以弥补LLM在实时信息、精确计算等方面的局限。
3.  **记忆**：构建多层次记忆系统，包括受限于上下文窗口的短期工作记忆，以及通过外部向量数据库实现的长期记忆，以支持持续学习与信息持久化。
4.  **响应生成**：整合规划、工具调用结果与记忆信息，生成最终答案。

**二、 关键能力与典型应用**
1.  **能力范围**：智能体可执行多样化操作，包括信息检索、文件处理、代码生成与执行、项目管理、内容创作及协调子代理等。
2.  **应用案例**：
    *   **代码生成**（如GPT-Engineer）：根据自然语言描述生成完整、结构化的软件项目。
    *   **科学发现**（如ChemCrow）：在专业领域（如化学）集成专家工具，自主规划并执行复杂实验流程。
    *   **社会模拟**（Generative Agents）：驱动虚拟角色产生信息传播、关系建立等涌现社会行为。
    *   **自主代理**（如AutoGPT）：展示了LLM自主决策、管理任务和利用资源的概念验证。

**三、 面临的核心挑战与风险**
1.  **技术瓶颈**：
    *   **效率与延迟**：LLM的多轮推理和外部工具调用导致响应速度慢。
    *   **上下文长度限制**：有限的上下文窗口是容纳历史、指令和复杂过程的主要瓶颈。
    *   **规划与可靠性**：在长期任务规划、应对意外及调整策略方面能力较弱；自然语言接口的输出不稳定（格式错误、指令遵循失败）增加了系统集成的复杂性。
    *   **评估局限**：在专业领域，LLM的自我评估与人类专家评估存在显著差距。
2.  **安全与风险**：存在被诱导执行高风险任务（如合成危险物质）的潜在滥用风险，凸显了强化安全边界与内容过滤的紧迫性。

**四、 前沿方向与设计原则**
1.  **研究进展**：前沿探索包括从反馈中学习的机制（如Chain of Hindsight）、从历史中学习策略的算法蒸馏（Algorithm Distillation），以及提升工具调用精准性、记忆检索效率等方法。
2.  **设计原则**：强调**自主性、效率、结构化与模块化**。通过鼓励自我反思、策略优化以及采用明确的指令集与架构模式，旨在以最少步骤高效、可靠地完成任务。

**总结**：LLM驱动的自主智能体正从强大的文本生成器演变为能够自主规划、持续学习并与环境交互的通用问题解决者，极大地扩展了AI的应用潜力。然而，其发展仍受限于效率、可靠性、安全性及复杂任务评估等严峻挑战。未来的突破需在增强核心能力的同时，着力于系统稳定性、风险控制与工程化落地。